# CNN (LeCun 1989) — Implementation from Scratch
# CNN (LeCun 1989) — 직접 구현

**Paper / 논문**: Y. LeCun et al., "Backpropagation Applied to Handwritten Zip Code Recognition" (1989)

이 노트북에서는 1989년 논문의 CNN 아키텍처를 NumPy로 직접 구현하고, MNIST 데이터셋에서 테스트합니다.

This notebook implements the 1989 CNN architecture from scratch in NumPy and tests on MNIST.

**Contents / 목차:**
1. **Part 1**: 2D Convolution from Scratch — 합성곱 연산 직접 구현
2. **Part 2**: CNN Architecture — 논문의 아키텍처 구현 (16×16 input → H1 → H2 → H3 → Output)
3. **Part 3**: Weight Sharing Visualization — 가중치 공유의 효과 시각화
4. **Part 4**: FC vs CNN Comparison — 논문 핵심 실험: Fully Connected vs CNN
5. **Part 5**: Learned Kernels — 학습된 커널 시각화 (Hubel & Wiesel 유사성)
6. **Part 6**: Parameter Count Analysis — 64,660 연결 vs 9,760 파라미터
7. **Part 7**: Modern PyTorch CNN — 현대 프레임워크로 재구현

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

## Part 1: 2D Convolution from Scratch / 합성곱 연산 직접 구현

CNN의 핵심 연산인 2D 합성곱을 NumPy로 구현합니다. 논문에서: "feature map이 수행하는 함수는 **5×5 커널을 가진 비선형 부분표본화 합성곱**으로 해석할 수 있다."

We implement the core CNN operation — 2D convolution — in NumPy. From the paper: "The function performed by a feature map can thus be interpreted as a **nonlinear subsampled convolution with a 5 by 5 kernel**."

In [ ]:
def conv2d(image: np.ndarray, kernel: np.ndarray, stride: int = 1,
           padding: float = -1.0) -> np.ndarray:
    """2D convolution with stride and constant padding.

    Implements the paper's convolution: "connections extending past the
    boundaries of the input plane take their input from a virtual background
    plane whose state is equal to a constant, predetermined background level."

    Args:
        image: 2D input array (H, W).
        kernel: 2D kernel array (kH, kW).
        stride: Step size for sliding the kernel (paper uses 2).
        padding: Constant value for out-of-bounds (paper uses -1).

    Returns:
        2D output feature map.
    """
    h, w = image.shape
    kh, kw = kernel.shape

    # Pad the image with constant value
    pad_h, pad_w = kh // 2, kw // 2
    padded = np.full((h + 2 * pad_h, w + 2 * pad_w), padding)
    padded[pad_h:pad_h + h, pad_w:pad_w + w] = image

    # Compute output size
    out_h = (h + 2 * pad_h - kh) // stride + 1
    out_w = (w + 2 * pad_w - kw) // stride + 1
    output = np.zeros((out_h, out_w))

    for i in range(out_h):
        for j in range(out_w):
            region = padded[i * stride:i * stride + kh,
                            j * stride:j * stride + kw]
            output[i, j] = np.sum(region * kernel)

    return output


def scaled_tanh(x: np.ndarray, a: float = 1.7159, s: float = 2/3) -> np.ndarray:
    """Scaled hyperbolic tangent activation (paper's choice over sigmoid).

    f(x) = A * tanh(S * x)
    Symmetric about origin → faster convergence than sigmoid.
    """
    return a * np.tanh(s * x)


# Demo: apply convolution with different kernels to a synthetic image
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Create a simple digit-like image (16x16)
image = np.zeros((16, 16))
# Draw a rough "7"
image[2, 3:12] = 1.0  # top horizontal
image[3:13, 10] = 1.0  # vertical stroke
for i in range(3, 13):
    image[i, max(3, 10 - (i - 3) // 2)] = 0.5  # diagonal

# Define feature detection kernels
kernels = {
    "Horizontal edge": np.array([[-1, -1, -1, -1, -1],
                                  [-1, -1, -1, -1, -1],
                                  [ 0,  0,  0,  0,  0],
                                  [ 1,  1,  1,  1,  1],
                                  [ 1,  1,  1,  1,  1]]) / 5,
    "Vertical edge": np.array([[-1, -1, 0, 1, 1],
                                [-1, -1, 0, 1, 1],
                                [-1, -1, 0, 1, 1],
                                [-1, -1, 0, 1, 1],
                                [-1, -1, 0, 1, 1]]) / 5,
    "Center-surround": np.array([[-1, -1, -1, -1, -1],
                                  [-1,  2,  2,  2, -1],
                                  [-1,  2,  8,  2, -1],
                                  [-1,  2,  2,  2, -1],
                                  [-1, -1, -1, -1, -1]]) / 16,
}

# Original image
axes[0, 0].imshow(image, cmap="gray", vmin=-1, vmax=1)
axes[0, 0].set_title("Original 16×16\n(원본 이미지)")
axes[1, 0].imshow(image, cmap="gray", vmin=-1, vmax=1)
axes[1, 0].set_title("Original")

for idx, (name, kernel) in enumerate(kernels.items()):
    # Without stride (full resolution)
    result_s1 = conv2d(image, kernel, stride=1)
    axes[0, idx + 1].imshow(result_s1, cmap="RdBu_r")
    axes[0, idx + 1].set_title(f"{name}\nstride=1, output {result_s1.shape}")

    # With stride 2 (paper's subsampling)
    result_s2 = conv2d(image, kernel, stride=2)
    activated = scaled_tanh(result_s2)
    axes[1, idx + 1].imshow(activated, cmap="RdBu_r")
    axes[1, idx + 1].set_title(f"{name}\nstride=2 + tanh, output {activated.shape}")

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("2D Convolution: Feature Detection Kernels\n"
             "(합성곱: 특징 감지 커널)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("Key observation from the paper:")
print("'Some kernels synthesized by the network can be interpreted as")
print(" feature detectors remarkably similar to those found in biological")
print(" vision systems (Hubel and Wiesel 1962).'")

## Part 2–6: Parameter Count & FC vs CNN / 파라미터 수 분석과 FC vs CNN 비교

논문의 가장 중요한 결과를 재현합니다: **연결 수가 아니라 자유 파라미터 수가 일반화를 결정**합니다.

We reproduce the paper's most important result: **free parameter count, not connection count, determines generalization.**

In [ ]:
# Reproduce the paper's parameter count analysis (Table from §3.3)

print("=" * 70)
print("LeCun 1989 CNN: Parameter Count Analysis")
print("(논문의 파라미터 수 분석 재현)")
print("=" * 70)

# H1: 12 feature maps × 8×8
h1_maps = 12
h1_size = 8  # 8×8
h1_kernel = 5  # 5×5
h1_units = h1_maps * h1_size * h1_size  # 768
h1_weights_per_unit = h1_kernel * h1_kernel + 1  # 25 + 1 bias = 26
h1_connections = h1_units * h1_weights_per_unit  # 768 × 26 = 19,968
h1_free_params = h1_maps * h1_weights_per_unit  # 12 × 26 = 312
# Paper says 1,068 (768 biases + 25×12 weights) — biases are NOT shared
h1_free_params_paper = h1_maps * h1_kernel * h1_kernel + h1_units  # 300 + 768
# Actually paper: "only 1068 free parameters (768 biases plus 25 times 12
# feature kernels)"
h1_free_params_paper = h1_units + h1_maps * h1_kernel * h1_kernel
# = 768 + 300 = 1068

print(f"\nLayer H1:")
print(f"  Feature maps: {h1_maps} × {h1_size}×{h1_size} = {h1_units} units")
print(f"  Kernel: {h1_kernel}×{h1_kernel}, stride 2")
print(f"  Connections: {h1_units} × {h1_weights_per_unit} = {h1_connections:,}")
print(f"  Free parameters: {h1_free_params_paper:,}")
print(f"    (biases per unit: {h1_units}, shared kernels: "
      f"{h1_maps}×{h1_kernel}²={h1_maps * h1_kernel**2})")
print(f"  Compression: {h1_free_params_paper/h1_connections:.1%}")

# H2: 12 feature maps × 4×4
h2_maps = 12
h2_size = 4
h2_input_maps = 8  # each H2 map takes from 8 H1 maps
h2_units = h2_maps * h2_size * h2_size  # 192
h2_weights_per_unit = h2_input_maps * h1_kernel * h1_kernel + 1  # 200+1=201
h2_connections = h2_units * h2_weights_per_unit  # 192 × 201 = 38,592
h2_free_params_paper = h2_maps * (h2_input_maps * h1_kernel**2) + h2_units
# = 12 × 200 + 192 = 2,592

print(f"\nLayer H2:")
print(f"  Feature maps: {h2_maps} × {h2_size}×{h2_size} = {h2_units} units")
print(f"  Kernel: {h1_kernel}×{h1_kernel}×{h2_input_maps} maps, stride 2")
print(f"  Connections: {h2_units} × {h2_weights_per_unit} = {h2_connections:,}")
print(f"  Free parameters: {h2_free_params_paper:,}")
print(f"  Compression: {h2_free_params_paper/h2_connections:.1%}")

# H3: 30 fully connected
h3_units = 30
h3_connections = h3_units * (h2_units + 1)  # 30 × 193 = 5,790
h3_free = h3_connections

print(f"\nLayer H3 (fully connected):")
print(f"  Units: {h3_units}")
print(f"  Connections = Free parameters: {h3_connections:,}")

# Output: 10 fully connected
out_units = 10
out_connections = out_units * (h3_units + 1)  # 10 × 31 = 310
out_free = out_connections

print(f"\nOutput:")
print(f"  Units: {out_units}")
print(f"  Connections = Free parameters: {out_connections:,}")

# Totals
total_units = 256 + h1_units + h2_units + h3_units + out_units
total_connections = h1_connections + h2_connections + h3_connections + out_connections
total_free = h1_free_params_paper + h2_free_params_paper + h3_free + out_free

print(f"\n{'='*70}")
print(f"TOTALS:")
print(f"  Units: {total_units:,}")
print(f"  Connections: {total_connections:,}")
print(f"  Free parameters: {total_free:,}")
print(f"  Compression ratio: {total_free/total_connections:.1%}")
print(f"  (Paper reports: 1,256 units, 64,660 connections, 9,760 params)")
print(f"{'='*70}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: connections vs parameters per layer
layers = ["H1\n(conv)", "H2\n(conv)", "H3\n(FC)", "Output\n(FC)"]
connections = [h1_connections, h2_connections, h3_connections, out_connections]
free_params = [h1_free_params_paper, h2_free_params_paper, h3_free, out_free]

x = np.arange(len(layers))
width = 0.35
bars1 = axes[0].bar(x - width/2, connections, width, label="Connections",
                      color="steelblue", edgecolor="black")
bars2 = axes[0].bar(x + width/2, free_params, width, label="Free parameters",
                      color="salmon", edgecolor="black")
axes[0].set_xlabel("Layer")
axes[0].set_ylabel("Count")
axes[0].set_title("Connections vs Free Parameters per Layer\n"
                   "(층별 연결 수 vs 자유 파라미터 수)")
axes[0].set_xticks(x)
axes[0].set_xticklabels(layers)
axes[0].legend()
axes[0].set_yscale("log")

# Right: FC vs CNN comparison (paper's key result)
models = ["FC Network\n(paper §5)", "CNN\n(paper §5)"]
model_connections = [10690, 64660]
model_params = [10690, 9760]
model_test_error = [8.1, 5.0]

ax2 = axes[1]
x2 = np.arange(len(models))
bars_conn = ax2.bar(x2 - width/2, model_params, width,
                     label="Free parameters", color="salmon", edgecolor="black")
bars_err = ax2.bar(x2 + width/2, [e * 1000 for e in model_test_error], width,
                    label="Test error (×1000)", color="#4daf4a", edgecolor="black")

# Annotate
for bar, val in zip(bars_conn, model_params):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
             f"{val:,}", ha="center", fontsize=11, fontweight="bold")
for bar, val in zip(bars_err, model_test_error):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1000 + 200,
             f"{val}%", ha="center", fontsize=11, fontweight="bold", color="green")

ax2.set_title("FC vs CNN: Paper's Key Result\n"
              "(FC vs CNN: 논문의 핵심 결과)")
ax2.set_xticks(x2)
ax2.set_xticklabels(models)
ax2.legend()

plt.tight_layout()
plt.show()

print("\nPaper's key message:")
print("CNN has 6× MORE connections but FEWER free parameters → BETTER generalization!")
print("'It is the number of free parameters, not connections, that matters.'")

## Part 7: Modern PyTorch CNN / 현대 프레임워크로 재구현

논문의 아키텍처를 PyTorch로 재구현하고 MNIST에서 테스트합니다.
1989년 9,760 파라미터 네트워크가 현대 프레임워크에서 얼마나 잘 작동하는지 확인합니다.

Reimplementing the paper's architecture in PyTorch and testing on MNIST.
Checking how well the 1989 9,760-parameter network works in a modern framework.

| 1989 (이 논문) | 2024 (PyTorch) |
|---|---|
| 16×16 grayscale | `nn.Conv2d` + `F.interpolate` |
| 5×5 kernel, stride 2 | `nn.Conv2d(kernel_size=5, stride=2)` |
| Scaled tanh $A\tanh(Sx)$ | `nn.Tanh()` or `nn.ReLU()` |
| SN simulator on SUN-4/260 | GPU in milliseconds |
| 3 days training | ~30 seconds |

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim
    from torchvision import datasets, transforms

    # LeCun 1989 CNN architecture in PyTorch
    class LeCun1989CNN(nn.Module):
        """CNN architecture from LeCun et al. (1989).

        Input: 16×16 grayscale image
        H1: 12 feature maps, 5×5 kernel, stride 2 → 8×8
        H2: 12 feature maps, 5×5 kernel, stride 2 → 4×4
        H3: 30 fully connected units
        Output: 10 units (digits 0-9)
        """

        def __init__(self):
            super().__init__()
            # H1: 1 input channel → 12 feature maps, 5×5 kernel, stride 2
            self.conv1 = nn.Conv2d(1, 12, kernel_size=5, stride=2, padding=2)
            # H2: 12 → 12 feature maps (paper uses 8 of 12, simplified here)
            self.conv2 = nn.Conv2d(12, 12, kernel_size=5, stride=2, padding=2)
            # H3: fully connected
            self.fc1 = nn.Linear(12 * 4 * 4, 30)
            # Output
            self.fc2 = nn.Linear(30, 10)

        def forward(self, x):
            x = torch.tanh(self.conv1(x))   # → (B, 12, 8, 8)
            x = torch.tanh(self.conv2(x))   # → (B, 12, 4, 4)
            x = x.view(x.size(0), -1)       # → (B, 192)
            x = torch.tanh(self.fc1(x))     # → (B, 30)
            x = self.fc2(x)                 # → (B, 10)
            return x

    # Count parameters
    model = LeCun1989CNN()
    total_params = sum(p.numel() for p in model.parameters())
    print(f"LeCun 1989 CNN (PyTorch): {total_params:,} parameters")
    print(f"Paper reports: 9,760 parameters\n")

    # Load MNIST, resize to 16×16
    transform = transforms.Compose([
        transforms.Resize((16, 16)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),  # Scale to [-1, 1]
    ])

    train_dataset = datasets.MNIST("./data", train=True, download=True,
                                    transform=transform)
    test_dataset = datasets.MNIST("./data", train=False, transform=transform)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64,
                                                shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1000)

    # Train
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    criterion = nn.CrossEntropyLoss()

    train_losses = []
    test_accs = []

    for epoch in range(10):
        model.train()
        epoch_loss = 0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            output = model(batch_x)
            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        # Test accuracy
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                output = model(batch_x)
                pred = output.argmax(dim=1)
                correct += (pred == batch_y).sum().item()
                total += batch_y.size(0)

        acc = correct / total
        train_losses.append(epoch_loss / len(train_loader))
        test_accs.append(acc)
        print(f"Epoch {epoch+1:2d}: loss={train_losses[-1]:.4f}, "
              f"test acc={acc:.2%}, test error={1-acc:.2%}")

    # Visualize learned kernels
    fig, axes = plt.subplots(2, 6, figsize=(15, 6))

    conv1_weights = model.conv1.weight.detach().numpy()
    for i in range(12):
        ax = axes[i // 6, i % 6]
        kernel = conv1_weights[i, 0]
        ax.imshow(kernel, cmap="RdBu_r")
        ax.set_title(f"Kernel {i+1}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])

    plt.suptitle("Learned H1 Kernels (cf. Hubel & Wiesel feature detectors)\n"
                 "(학습된 H1 커널 — 생물학적 특징 감지기와 비교)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # Final comparison
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot(train_losses, "b-o", linewidth=2)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Training Loss")

    axes[1].plot([1 - a for a in test_accs], "r-o", linewidth=2)
    axes[1].axhline(y=0.05, color="green", linestyle="--",
                     label="Paper result: 5.0%")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Test Error Rate")
    axes[1].set_title("Test Error Rate\n(테스트 오류율)")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    print(f"\nFinal test error: {1 - test_accs[-1]:.2%}")
    print(f"Paper's test error on zip codes: 5.0%")
    print(f"Note: MNIST is 'cleaner' than real zip codes, so we expect better results.")

except ImportError:
    print("PyTorch or torchvision not installed. Skipping.")
    print("Install with: pip install torch torchvision")
    print("\nThe key point: the 1989 architecture with ~10K parameters")
    print("still works remarkably well on digit recognition in 2024.")